In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

for f in ['dataset_m1_train.jsonl', 'dataset_m2_train.jsonl', 'dataset_m3_train.jsonl']:
    p = Path(DRIVE_PATH + f)
    print(p, '→', '✅ exists' if p.exists() else '❌ NOT FOUND')

In [ ]:
import json
import re
import math
from pathlib import Path
from google.colab import drive

# ── Syllable counter (French) ──────────────────────────────────────
def count_syllables_fr(word):
    word = word.lower().strip(".,!?;:\"'«»()-")
    if not word:
        return 0
    # Remove silent 'e' at end
    if len(word) > 2 and word.endswith('e') and word[-2] not in 'aeiouéèêëàâùûîïôœ':
        word = word[:-1]
    vowels = 'aeiouyéèêëàâùûîïôœæ'
    count = 0
    prev_vowel = False
    for ch in word:
        is_vowel = ch in vowels
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel
    return max(1, count)

# ── Sentence splitter ──────────────────────────────────────────────
def split_sentences(text):
    sentences = re.split(r'[.!?]+', text)
    return [s.strip() for s in sentences if s.strip()]

# ── Word tokenizer ─────────────────────────────────────────────────
def tokenize_words(text):
    # Remove syllabic hyphens (ma-man → maman) before counting
    text = re.sub(r'(?<=[a-zA-ZÀ-ÿ])-(?=[a-zA-ZÀ-ÿ])', '', text)
    return re.findall(r"[a-zA-ZÀ-ÿ']+", text.lower())

# ── Simple word list (≤2 syllables = simple) ──────────────────────
def is_simple_word(word):
    return count_syllables_fr(word) <= 2

# ── Metrics ───────────────────────────────────────────────────────
def flesch_fr(text):
    """Flesch Reading Ease adapted for French (Kandel & Moles formula)"""
    sentences = split_sentences(text)
    words = tokenize_words(text)
    if not sentences or not words:
        return 0
    asl = len(words) / len(sentences)          # avg sentence length
    asw = sum(count_syllables_fr(w) for w in words) / len(words)  # avg syllables/word
    score = 207 - (1.015 * asl) - (73.6 * asw)
    return round(score, 2)

def gunning_fog(text):
    sentences = split_sentences(text)
    words = tokenize_words(text)
    if not sentences or not words:
        return 0
    complex_words = [w for w in words if count_syllables_fr(w) >= 3]
    asl = len(words) / len(sentences)
    pcw = len(complex_words) / len(words) * 100
    return round(0.4 * (asl + pcw), 2)

def ari(text):
    text = re.sub(r'(?<=[a-zA-ZÀ-ÿ])-(?=[a-zA-ZÀ-ÿ])', '', text)  # strip syllabic hyphens
    sentences = split_sentences(text)
    words = tokenize_words(text)
    chars = len(re.sub(r'\s', '', text))
    if not sentences or not words:
        return 0
    return round(4.71 * (chars / len(words)) + 0.5 * (len(words) / len(sentences)) - 21.43, 2)

def avg_sentence_length(text):
    sentences = split_sentences(text)
    words = tokenize_words(text)
    if not sentences:
        return 0
    return round(len(words) / len(sentences), 2)

def simple_word_ratio(text):
    words = tokenize_words(text)
    if not words:
        return 0
    simple = [w for w in words if is_simple_word(w)]
    return round(len(simple) / len(words), 4)

# ── Full metric bundle ─────────────────────────────────────────────
def compute_metrics(text):
    return {
        'flesch':        flesch_fr(text),
        'gunning_fog':   gunning_fog(text),
        'ari':           ari(text),
        'avg_sent_len':  avg_sentence_length(text),
        'simple_ratio':  simple_word_ratio(text),
        'n_sentences':   len(split_sentences(text)),
        'n_words':       len(tokenize_words(text)),
    }

# ── Run on your dataset ────────────────────────────────────────────
def evaluate_readability(jsonl_path):
    results = []
    with open(jsonl_path) as f:
        for line in f:
            ex = json.loads(line)
            orig = ex.get('input', '')
            simp = ex.get('output', '')
            results.append({
                'id':       ex.get('id', '?'),
                'niveau':   ex.get('niveau', '?'),
                'original': compute_metrics(orig),
                'simplified': compute_metrics(simp),
            })
    return results

def print_summary(results):
    def avg(key, group):
        vals = [r[group][key] for r in results if r[group][key] != 0]
        return round(sum(vals) / len(vals), 2) if vals else 0

    metrics = ['flesch', 'gunning_fog', 'ari', 'avg_sent_len', 'simple_ratio']
    print(f"{'Metric':<20} {'Original':>10} {'Simplified':>12} {'Δ':>8}")
    print("-" * 54)
    for m in metrics:
        o = avg(m, 'original')
        s = avg(m, 'simplified')
        delta = round(s - o, 2)
        arrow = '↑' if (m == 'flesch' or m == 'simple_ratio') and delta > 0 else \
                '↓' if (m != 'flesch' and m != 'simple_ratio') and delta < 0 else '→'
        print(f"{m:<20} {o:>10} {s:>12} {arrow} {delta:>6}")

# ── Run ───────────────────────────────────────────────────────────
DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

for dataset, label in [
    ('dataset_m1_train.jsonl', 'Model 1 (99 pairs)'),
    ('dataset_m2_train.jsonl', 'Model 2 (243 pairs)'),
    ('dataset_m3_train.jsonl', 'Model 3 (294 pairs)'),
]:
    path = DRIVE_PATH + dataset
    if Path(path).exists():
        results = evaluate_readability(path)
        print(f'\n{"="*54}')
        print(f'  {label}')
        print(f'{"="*54}')
        print_summary(results)